# 06 L2 Regularization

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb008_s3_alpha.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_alpha.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s3_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_alpha.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'ddca47686ea53c3ea3bedd2289fc126604c7c460'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'e77f683caff97da3118a1d8ad21056c8b31067c6'}}

## exp_cmb008_s3_alpha

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_alpha.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_alpha.py)  
Git blob: `ddca47686ea53c3ea3bedd2289fc126604c7c460`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 / S3. Freeze (256,) and vary only alpha (L2 penalty).

S3 alpha 1e-5, 1e-3, 1e-2 use the exact S1 preprocessor and train/heldout
sources; archived S1 (256,) alpha=1e-4 is the baseline, no duplicate fit.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.

CLI 인자 (정적 추출): `--arm`, `--alpha`, `--seed`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_alpha.py
#!/usr/bin/env python3
"""EXP-CMB-008 / S3. Freeze (256,) and vary only alpha (L2 penalty).

S3 alpha 1e-5, 1e-3, 1e-2 use the exact S1 preprocessor and train/heldout
sources; archived S1 (256,) alpha=1e-4 is the baseline, no duplicate fit.
3 seeds (42/43/44) and 4 writer-held-out folds per candidate.
No Original/Professor Test or new 480. No held-out/OfficialVal training augmentation.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,sys,time,warnings
from pathlib import Path
import joblib
import numpy as np
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,roc_auc_score,
                             precision_recall_fscore_support,confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT=Path(__file__).resolve().parents[4]
CMB=ROOT/"05_model_optimize"/"MLP"/"Combined"
NORMAL=ROOT/"05_model_optimize"/"MLP"/"Normal"
sys.path.insert(0,str(CMB/"scripts"))
sys.path.insert(0,str(NORMAL/"scripts"))
from combined_mlp_writer_cv import (load_official,load_writers,PixelScaler255,
                                    WRITERS,LABELS,V3_SHA,HANDMADE_SHA)
from exp006_size_com_comparison import process,image_geometry
from exp015_scale_stroke_grid import nested_subset
from exp016_high_fraction_grid import scale_base,stroke_jitter
from exp014_balanced_augmentation import balanced_indices
from exp011_012_causal_probes import fixed_scale

SCALES=(.8,.9,1.,1.1,1.2)
PAIRS=(("0","D"),("D","0"),("8","B"),("B","8"),("5","F"),("F","5"),
       ("7","9"),("7","A"),("4","F"))
ARMS={"com_only":(0.,0.),
      "scale10_stroke00":(.1,0.),
      "scale10_stroke14":(.1,.14),
      "scale10_stroke28":(.1,.28),
      "scale20_stroke00":(.2,0.),
      "scale20_stroke14":(.2,.14),
      "scale20_stroke28":(.2,.28),
      "scale30_stroke00":(.3,0.),
      "scale30_stroke14":(.3,.14),
      "scale30_stroke28":(.3,.28)}
TARGETS=(16,18,20,22,24)
SEEDS=(42,43,44)
CLASSES=np.arange(16,dtype=int)

def savecsv(p,records):
    records=list(records)
    if not records:raise RuntimeError("missing rows "+str(p))
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        wr=csv.DictWriter(f,fieldnames=list(records[0]))
        wr.writeheader();wr.writerows(records)

def metrics(y,p,subset=None):
    labels=CLASSES if subset is None else np.asarray(subset)
    prec,rec,f1,sup=precision_recall_fscore_support(y,p,labels=labels,zero_division=0)
    return {"n":int(len(y)),"correct":int(np.sum(p==y)),
            "accuracy":float(np.mean(p==y)),
            "macro_f1":float(f1.mean()),
            "macro_precision":float(prec.mean()),
            "macro_recall":float(rec.mean())}

def confusion_pairs(y,p,writer,arm,seed,source):
    """All 240 off-diagonal directions, including zero-error pairs.

    Denominator n_actual corresponds to the *actual* (row) class.
    Both counts and conditional percentages are saved; total overall
    accuracy does not replace per-class and per-pair error analysis.
    """
    cm=confusion_matrix(y,p,labels=CLASSES)
    result=[]
    nall=int(len(y))
    nwrong=int(nall-np.trace(cm))
    for actual in CLASSES:
        nactual=int(cm[actual].sum())
        wrong=int(nactual-cm[actual,actual])
        for target in CLASSES:
            if actual==target:continue
            ct=int(cm[actual,target])
            result.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
                 "actual":LABELS[actual],"predicted":LABELS[target],
                 "n_actual":nactual,"n_actual_wrong":wrong,
                 "n_dataset":nall,"n_dataset_wrong":nwrong,
                 "errors":ct,"error_rate_within_actual_class":ct/nactual if nactual else 0,
                 "share_of_actual_class_errors":ct/wrong if wrong else 0,
                 "share_of_all_dataset_errors":ct/nwrong if nwrong else 0})
    assert len(result)==16*15
    assert sum(int(r["errors"]) for r in result)==nwrong
    return result

def total_errors_by_class(y,p,writer,arm,seed,source):
    """All 16 true classes: total incorrect / n actual class, plus group switch."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    out=[]
    for a in CLASSES:
        n=int(cm[a].sum());correct=int(cm[a,a]);wrong=n-correct
        family_swap=int(cm[a,10:].sum()) if a<10 else int(cm[a,:10].sum())
        out.append({"arm":arm,"seed":seed,"dataset":source,"writer":writer,
           "actual":LABELS[a],"n":n,"correct":correct,"wrong":wrong,
           "class_error_rate":wrong/n if n else 0,
           "opposite_family_errors":family_swap,
           "opposite_family_error_rate":family_swap/n if n else 0})
    assert len(out)==16
    return out

def family_error_breakdown(y,p,writer,arm,seed,source):
    """Digits→letters and letters→digits alongside within-family mistakes."""
    cm=confusion_matrix(y,p,labels=CLASSES)
    d2l=int(cm[:10,10:].sum());l2d=int(cm[10:,:10].sum())
    d2d=int(cm[:10,:10].sum()-np.trace(cm[:10,:10]))
    l2l=int(cm[10:,10:].sum()-np.trace(cm[10:,10:]))
    return [{"arm":arm,"seed":seed,"dataset":source,"writer":writer,
      "direction":key,"errors":val,"n_actual_family":n,
      "error_rate":val/n if n else 0}
      for key,val,n in (("digit_to_letter",d2l,int(cm[:10].sum())),
                        ("letter_to_digit",l2d,int(cm[10:].sum())),
                        ("digit_to_digit",d2d,int(cm[:10].sum())),
                        ("letter_to_letter",l2l,int(cm[10:].sum())))]

def geom_rows(name,x,y,arm,seed):
    geo=image_geometry(x)
    out=[]
    for nameg,mask in (("digits",y<10),("letters",y>=10)):
        z={"arm":arm,"seed":seed,"dataset":name,"group":nameg,
           "n":int(mask.sum())}
        for field in ("long_side","width","height","mass"):
            zz=np.asarray(geo[field])[mask]
            z[field+"_mean"]=float(zz.mean())
            z[field+"_std"]=float(zz.std())
        out.append(z)
    return out

def binary_size_probe(xtrain,ytrain,xval,yval):
    train=image_geometry(xtrain)["long_side"].reshape(-1,1)
    val=image_geometry(xval)["long_side"].reshape(-1,1)
    model=Pipeline([("std",StandardScaler()),
      ("logistic",LogisticRegression(class_weight="balanced",random_state=42,max_iter=1000))])
    model.fit(train,(ytrain>=10).astype(int))
    q=model.predict_proba(val)[:,1]
    return {"bal_acc":float(balanced_accuracy_score(yval>=10,q>=.5)),
            "auc":float(roc_auc_score(yval>=10,q))}

def prepareraw(official,writers,arm):
    ratio,stroke=ARMS[arm]
    raw=official["X_train"]
    yy=official["y_train"]
    # CoM-only is exactly the EXP-CMB-004 reference condition.
    main=process(raw,"com",20)
    changed={"method":arm,"requested_ratio":ratio,"actual_ratio":0.,
             "n_aug":0,"n_stroke_changed":0,"n_stroke_rejected":0,
             "official_train_rows":len(raw)}
    targets=[]
    if ratio:
        ids,target_map=nested_subset(yy,ratio)  # Normal EXP015 nested low-ratio group
        half,_=balanced_indices(yy,seed=20261116)
        augmented=scale_base(raw,ids,target_map)
        augmented,stroke_stats=stroke_jitter(augmented,stroke,ids,half)
        main[ids]=process(augmented,"com",20)
        changed.update({"n_aug":len(ids),"actual_ratio":len(ids)/len(yy),
             "n_stroke_changed":stroke_stats["modified"],
             "n_stroke_rejected":stroke_stats["rejected"]})
        for cl in CLASSES:
            for length in TARGETS:
                ct=int(np.sum((yy[ids]==cl)&(target_map[ids]==length)))
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":ct})
                assert ct>=1
        for cl in CLASSES:
            cc=[x["train_images_assigned"] for x in targets if x["class"]==LABELS[cl]]
            assert max(cc)-min(cc)<=1
        assert abs(len(ids)/len(yy)-ratio)<.002
    else:
        for cl in CLASSES:
            for length in TARGETS:
                targets.append({"arm":arm,"class":LABELS[cl],"target_size":length,
                                "train_images_assigned":0})
    transformed_w={w:process(writers[w]["X"],"com",20) for w in WRITERS}
    assert main.shape==(76800,784) and len(transformed_w)==4
    return main,transformed_w,changed,targets

def make_scaled(official,writers):
    ref={}
    for scale in SCALES:
        # exact Normal EXP012 intervention; raw image BBox-centered affine, then CoM
        ref[("official",scale)]=process(fixed_scale(official["X_val"],scale),"com",20)
        for w in WRITERS:
            ref[(w,scale)]=process(fixed_scale(writers[w]["X"],scale),"com",20)
    return ref

def eval_scales(model,raw_map,y,dataset,writer,arm,seed):
    rows=[]; pair_rows=[]; focus=[]; class_errors=[]; family_errors=[]
    probs={}
    for sc in SCALES:
        xx=raw_map[(writer if dataset=="OOF" else "official",sc)]
        q=np.asarray(model.predict_proba(xx),dtype=np.float64)
        assert np.array_equal(model.named_steps["model"].classes_,CLASSES)
        p=q.argmax(axis=1).astype(int)
        letter=q[:,10:].sum(axis=1)
        probs[sc]=(p,letter)
    _,base_letter=probs[1.]
    base_family=base_letter>=.5
    for sc in SCALES:
        p,letter=probs[sc]
        for group,z,cls in (("all",np.ones(len(y),dtype=bool),CLASSES),
             ("digits",y<10,np.arange(10)),("letters",y>=10,np.arange(10,16))):
            m=metrics(y[z],p[z],cls)
            flip=(letter[z]>=.5)!=(base_family[z])
            rows.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
               "scale":sc,"group":group,**m,
               "family_flip":float(flip.mean()),
               "mean_abs_delta_p_letter":float(np.abs(letter[z]-base_letter[z]).mean()),
               "mean_p_letter":float(letter[z].mean())})
        for cls_i in CLASSES:
            z=y==cls_i
            pz=p[z];ll=letter[z];bb=base_letter[z]
            focus.append({"arm":arm,"seed":seed,"writer":writer,"dataset":dataset,
                "scale":sc,"actual":LABELS[cls_i],"n":int(z.sum()),
                "correct":int(np.sum(pz==cls_i)),
                "accuracy":float(np.mean(pz==cls_i)),
                "family_flip":float(np.mean((ll>=.5)!=(bb>=.5))),
                "mean_abs_delta_p_letter":float(np.mean(np.abs(ll-bb))),
                "mean_p_letter":float(ll.mean())})
        sc_name=dataset+"_scale_"+str(sc)
        pair_rows+=confusion_pairs(y,p,writer,arm,seed,sc_name)
        class_errors+=total_errors_by_class(y,p,writer,arm,seed,sc_name)
        family_errors+=family_error_breakdown(y,p,writer,arm,seed,sc_name)
    return rows,pair_rows,focus,probs[1.][0],class_errors,family_errors

def output_plots(y,p,out,arm,seed):
    cm=confusion_matrix(y,p,labels=CLASSES)
    for k in ("counts","row_normalized"):
        v=cm if k=="counts" else 100*cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(10,8))
        im=ax.imshow(v,cmap="viridis",vmin=0,vmax=160 if k=="counts" else 100)
        fig.colorbar(im,ax=ax,fraction=.045,pad=.04)
        ax.set(xticks=np.arange(16),yticks=np.arange(16),
               xticklabels=list(LABELS),yticklabels=list(LABELS),
               xlabel="Predicted",ylabel="True",
               title=f"EXP-CMB-005 {arm} seed={seed} Combined OOF {k}")
        fig.tight_layout()
        fig.savefig(out/f"OOF_confusion_{k}.png",dpi=145)
        plt.close(fig)

def run(arm,seed,out,alpha):
    if arm!="scale30_stroke28" or seed not in SEEDS or alpha not in (1e-5,1e-3,1e-2):raise ValueError((arm,seed,alpha))
    if out.exists(): raise FileExistsError(out)
    out.mkdir(parents=True)
    (out/"models").mkdir()
    t0=time.perf_counter()
    original=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
    handmade=ROOT/"02_data"/"Hand_made"
    off=load_official(original)
    writers=load_writers(handmade)
    assert off["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    assert len(off["X_train"])==76800 and len(off["X_val"])==19200
    tpre=time.perf_counter()
    xtrain_off,xwriters,info,targets=prepareraw(off,writers,arm)
    xval=process(off["X_val"],"com",20)
    scale_map=make_scaled(off,writers)
    preprocess_time=time.perf_counter()-tpre
    official_probe=binary_size_probe(xtrain_off,off["y_train"],xval,off["y_val"])
    savecsv(out/"target_assignments.csv",targets)
    features=geom_rows("official_train",xtrain_off,off["y_train"],arm,seed)
    features+=geom_rows("official_val",xval,off["y_val"],arm,seed)
    for w in WRITERS:features+=geom_rows("writer_"+str(w),xwriters[w],writers[w]["y"],arm,seed)
    savecsv(out/"geometry_distributions.csv",features)
    foldrows=[];pairs=[];class_errors=[];family_errors=[];scale_rows=[];class_rows=[];scale_class_rows=[];pred_rows=[];loss=[];fit=[]
    pooled=[]
    for held in WRITERS:
        trwriters=[w for w in WRITERS if w!=held]
        X=np.concatenate([xtrain_off]+[xwriters[w] for w in trwriters])
        y=np.concatenate([off["y_train"]]+[writers[w]["y"] for w in trwriters])
        assert X.shape==(78720,784) and len(y)==78720
        mlp=Pipeline([("scale_0_1",PixelScaler255()),
                      ("model",MLPClassifier(random_state=seed,hidden_layer_sizes=(256,),alpha=alpha))])
        started=time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mlp.fit(X,y)
        sec=time.perf_counter()-started
        mlpnet=mlp.named_steps["model"]
        assert mlpnet.n_features_in_==784
        for it,lossval in enumerate(mlpnet.loss_curve_):
            loss.append({"arm":arm,"seed":seed,"heldout":held,
                        "iteration":it+1,"loss":float(lossval)})
        fit.append({"arm":arm,"seed":seed,"heldout":held,
                    "train_n":len(y),"heldout_n":640,
                    "fit_seconds":sec,"iterations":mlpnet.n_iter_,
                    "last_loss":float(mlpnet.loss_),
                    "warnings":" | ".join(str(w.message) for w in caught)})
        fout=out/"models"/f"heldout_{held}.joblib"
        joblib.dump(mlp,fout,compress=3)
        handwritten_y=writers[held]["y"]
        outcome=mlp.predict(xwriters[held]).astype(int)
        pp=mlp.predict(xval).astype(int)
        for ds,ytrue,pred in (("OOF",handwritten_y,outcome),
                              ("OfficialVal",off["y_val"],pp)):
            maskset=(("all",np.ones(len(ytrue),bool),CLASSES),
                     ("digits",ytrue<10,np.arange(10)),
                     ("letters",ytrue>=10,np.arange(10,16)))
            for g,z,classes in maskset:
                foldrows.append({"arm":arm,"seed":seed,"heldout":held,"dataset":ds,
                      "group":g,**metrics(ytrue[z],pred[z],classes)})
        prec,rec,f1,_=precision_recall_fscore_support(handwritten_y,outcome,
              labels=CLASSES,zero_division=0)
        for ci in CLASSES:
            z=handwritten_y==ci
            class_rows.append({"arm":arm,"seed":seed,"writer":held,
               "class":LABELS[ci],"n":int(z.sum()),"correct":int(np.sum(outcome[z]==ci)),
               "precision":float(prec[ci]),"recall":float(rec[ci]),"f1":float(f1[ci])})
        for i in range(640):
            pred_rows.append({"arm":arm,"seed":seed,"writer":held,"index":i,
                "actual":LABELS[int(handwritten_y[i])],
                "pred":LABELS[int(outcome[i])]})
        pooled.append(outcome)
        pairs+=confusion_pairs(handwritten_y,outcome,held,arm,seed,"OOF_original")
        pairs+=confusion_pairs(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        class_errors+=total_errors_by_class(handwritten_y,outcome,held,arm,seed,"OOF_original")
        class_errors+=total_errors_by_class(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        family_errors+=family_error_breakdown(handwritten_y,outcome,held,arm,seed,"OOF_original")
        family_errors+=family_error_breakdown(off["y_val"],pp,held,arm,seed,"OfficialVal_original")
        for ds,ytrue,wid in (("OOF",handwritten_y,held),
                             ("OfficialVal",off["y_val"],held)):
            z,p2,c,base,ce,fe=eval_scales(mlp,scale_map,ytrue,ds,wid,arm,seed)
            scale_rows+=z;pairs+=p2;scale_class_rows+=c
            class_errors+=ce;family_errors+=fe
            if ds=="OOF":assert np.array_equal(base,outcome)
            else:assert np.array_equal(base,pp)
        print("FOLD_COMPLETED",arm,seed,held,
              "OOF",round(np.mean(outcome==handwritten_y),6),
              "VAL",round(np.mean(pp==off["y_val"]),6),
              "fit_s",round(sec,2),flush=True)
        del X,y,mlp
    hand_y=np.concatenate([writers[w]["y"] for w in WRITERS])
    pooled_p=np.concatenate(pooled)
    assert len(hand_y)==2560
    output_plots(hand_y,pooled_p,out,arm,seed)
    savecsv(out/"fold_metrics.csv",foldrows)
    savecsv(out/"oof_per_class.csv",class_rows)
    savecsv(out/"per_class_and_scale.csv",scale_class_rows)
    savecsv(out/"pair_confusions.csv",pairs)
    savecsv(out/"per_class_error_rates.csv",class_errors)
    savecsv(out/"family_confusion_rates.csv",family_errors)
    savecsv(out/"scale_sensitivity.csv",scale_rows)
    savecsv(out/"oof_predictions.csv",pred_rows)
    savecsv(out/"training_loss.csv",loss)
    savecsv(out/"training_time.csv",fit)
    overall=metrics(hand_y,pooled_p)
    meta={"experiment":"EXP-CMB-008","stage":"S3_alpha","alpha":alpha,"hidden_layer_sizes":[256],"status":"S3 model trained, selection pending",
          "arm":arm,"seed":seed,"handmade_cv":"Leave-One-Writer-Out 4-Fold",
          "official_n":76800,"official_val_n":19200,"training_handmade_each_fold":1920,
          "oof_n":2560,"scale_interventions":SCALES,"class_balanced_targets":TARGETS,
          "official_train_only_augmentation":True,
          "handmade_train_preprocess":"CoM-only",
          "heldout_and_official_val_preprocess":"CoM-only",
          "augmentation_seed":20261116,"augmentation":info,
          "preregistration":"EXP-CMB-008 S3; width 256, only L2 alpha 1e-5/1e-3/1e-2; no test",
          "hyperparameter_search":True,"only_changed_parameter":"alpha",
          "fixed_params":{"activation":"relu","solver":"adam","hidden_layer_sizes":[256],"alpha":alpha,
             "learning_rate_init":0.001,"max_iter":200,"early_stopping":False},
          "official_sha256":off["sha256"],
          "handmade_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
          "official_train_size_binary_probe":official_probe,
          "oof_metrics":overall,"fit_total_seconds":sum(r["fit_seconds"] for r in fit),
          "preprocessing_seconds":preprocess_time,
          "wall_seconds":time.perf_counter()-t0,
          "original_test_used":False,"professor_test_used":False,
          "new_handmade_480_used":False,
          "versions":{"numpy":np.__version__,"scikit_learn":sklearn.__version__},
          "model_sha256":{str(w):hashlib.sha256((out/"models"/f"heldout_{w}.joblib").read_bytes()).hexdigest()
                          for w in WRITERS}}
    (out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("COMPLETED",arm,seed,json.dumps(overall),flush=True)

if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--arm",choices=("scale30_stroke28",),default="scale30_stroke28")
    ap.add_argument("--alpha",type=float,choices=(1e-5,1e-3,1e-2),required=True)
    ap.add_argument("--seed",type=int,choices=SEEDS,required=True)
    ap.add_argument("--out",type=Path,required=True)
    ar=ap.parse_args()
    run(ar.arm,ar.seed,ar.out,ar.alpha)


## exp_cmb008_s3_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_aggregate.py)  
Git blob: `e77f683caff97da3118a1d8ad21056c8b31067c6`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-008 S3: L2 alpha comparison, frozen (256,) architecture, no model refit or Test access.

Freeze (256,) after S2; alpha 1e-5, 1e-4, 1e-3, 1e-2.
Same 30% Size / 28% Stroke + CoM, seeds, writer folds and default other HP.
Prioritized gate: each actual-class directional D->0/B->8/8->B/0->D rate <5%,
otherwise best OOF accuracy without regression in those errors and guardrails.

CLI 인자 (정적 추출): `--new`, `--s1-reference`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s3_aggregate.py
#!/usr/bin/env python3
"""EXP-CMB-008 S3: L2 alpha comparison, frozen (256,) architecture, no model refit or Test access.

Freeze (256,) after S2; alpha 1e-5, 1e-4, 1e-3, 1e-2.
Same 30% Size / 28% Stroke + CoM, seeds, writer folds and default other HP.
Prioritized gate: each actual-class directional D->0/B->8/8->B/0->D rate <5%,
otherwise best OOF accuracy without regression in those errors and guardrails.
"""
import argparse,csv,json,sys
from pathlib import Path
from exp_cmb008_s1_aggregate import SEEDS,WRITERS,readcsv,summarize,check,target_under_5percent,paired,pct

ALPHAS=("1e-5","1e-4","1e-3","1e-2")
WATCH=("D->0","B->8","8->B","0->D")

def artifact(incoming,ref,alpha_key,seed):
    return ((ref/f"cmb008-w256-s{seed}") if alpha_key=="1e-4" else
            (incoming/f"cmb008-s3-a{alpha_key}-s{seed}"))

def read_alpha(incoming,ref,alpha_key):
    result={}
    for seed in SEEDS:
        p=artifact(incoming,ref,alpha_key,seed)
        assert p.is_dir(),p
        m=json.loads((p/"manifest.json").read_text(encoding="utf-8"))
        assert m["experiment"]=="EXP-CMB-008" and m["arm"]=="scale30_stroke28"
        assert int(m["seed"])==seed and m["augmentation_seed"]==20261116
        assert m["handmade_train_preprocess"]=="CoM-only"
        assert m["heldout_and_official_val_preprocess"]=="CoM-only"
        assert m["official_train_only_augmentation"]
        assert not any(m[k] for k in ("original_test_used","professor_test_used","new_handmade_480_used"))
        expected={"activation":"relu","solver":"adam","alpha":float(alpha_key),
                  "learning_rate_init":0.001,"max_iter":200,"early_stopping":False}
        if alpha_key=="1e-4":
            assert m["stage"]=="S1_width" and m["width"]==256
            assert m["only_changed_parameter"]=="hidden_layer_sizes"
        else:
            assert m["stage"]=="S3_alpha"
            assert m["hidden_layer_sizes"]==[256] and m["alpha"]==float(alpha_key)
            assert m["only_changed_parameter"]=="alpha"
            expected["hidden_layer_sizes"]=[256]
        assert m["fixed_params"]==expected
        oof=readcsv(p/"oof_predictions.csv")
        assert len(oof)==2560 and len({(int(x["writer"]),int(x["index"])) for x in oof})==2560
        vals=[x for x in readcsv(p/"fold_metrics.csv") if x["dataset"]=="OfficialVal" and x["group"]=="all"]
        assert len(vals)==4 and all(int(x["n"])==19200 for x in vals)
        scales=[x for x in readcsv(p/"per_class_and_scale.csv") if
                x["dataset"] in ("OOF","OfficialVal") and float(x["scale"]) in (.8,1.2)]
        assert scales
        times=readcsv(p/"training_time.csv")
        assert len(times)==4
        result[seed]={"m":m,"oof":oof,"val":vals,"scales":scales,"times":times}
    return result

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--new",type=Path,required=True)
    ap.add_argument("--s1-reference",type=Path,required=True)
    ap.add_argument("--out",type=Path,required=True)
    a=ap.parse_args()
    data={alpha_key:read_alpha(a.new,a.s1_reference,alpha_key) for alpha_key in ALPHAS}
    for alpha_key in ALPHAS:
        for seed in SEEDS:
            assert data[alpha_key][seed]["m"]["official_sha256"]==data["1e-4"][seed]["m"]["official_sha256"]
            assert data[alpha_key][seed]["m"]["handmade_sha256"]==data["1e-4"][seed]["m"]["handmade_sha256"]
    measured={alpha_key:summarize(alpha_key,data[alpha_key]) for alpha_key in ALPHAS}
    base=measured["1e-4"]
    assert abs(base["oof"]-.9221354166666667)<1e-9,base["oof"]
    assert abs(base["val"]-.9704427083333333)<1e-9,base["val"]
    checks={alpha_key:check(base,measured[alpha_key]) for alpha_key in ALPHAS}
    all_under=[alpha_key for alpha_key in ALPHAS if target_under_5percent(measured[alpha_key])]
    primary=[alpha_key for alpha_key in all_under if all(checks[alpha_key].values())]
    fallback=[alpha_key for alpha_key in ALPHAS if all(checks[alpha_key].values())]
    assert "1e-4" in fallback
    pool=primary or fallback
    bestacc=max(measured[k]["oof"] for k in pool)
    near=[k for k in pool if measured[k]["oof"]>=bestacc-.002-1e-12]
    near.sort(key=lambda k:(measured[k]["pairs"]["D->0"],
            sum(measured[k]["pairs"][q] for q in WATCH if q!="D->0"),
            -measured[k]["oof"],measured[k]["seed_std"],
            -measured[k]["val"],0 if k=="1e-4" else float(k)))
    winner=near[0]
    status="TARGET_MET" if primary else ("FALLBACK_IMPROVED" if winner!="1e-4" else "BASELINE_RETAINED")
    paired_metrics={alpha_key:paired(data["1e-4"],data[alpha_key]) for alpha_key in ALPHAS if alpha_key!="1e-4"}
    report={"experiment":"EXP-CMB-008","stage":"S3_alpha","status":status,
        "target_rate_each_direction":"strictly < 5%","denominator_per_actual_class":480,
        "strict_target_max_count":23,"priority_confusion":"D->0",
        "selected_alpha_for_S4":winner,"selected_is_final_hp":False,
        "primary_pass":primary,"safe_fallback_candidates":fallback,
        "data_sha_verified_against_s1":True,
        "new_fits":36,"s1_models_reused":12,
        "metrics":measured,"checks":checks,"paired":paired_metrics,
        "original_test_used":False,"professor_test_used":False,
        "selection_reason":"OOF accuracy first under <5% target; if not achievable, no-increase in four directions and protected classes plus Writer/Val/Flip safeguards"}
    a.out.mkdir(parents=True,exist_ok=True)
    (a.out/"S3_details.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    cols=["alpha","oof_accuracy","oof_error_rate","official_val_accuracy","seed_std",
          "D_to_0","D_to_0_rate","B_to_8","B_to_8_rate",
          "8_to_B","8_to_B_rate","0_to_D","0_to_D_rate",
          "class_0_error","class_8_error","class_B_error","class_D_error",
          "oof_worst_flip","official_val_worst_flip","mean_n_iter","mean_fit_seconds",
          "warning_folds","target_4_under_5pct","safeguards_pass","failed_gates"]
    rows=[]
    for alpha_key in ALPHAS:
        x=measured[alpha_key]
        c=checks[alpha_key]
        vals=[alpha_key,x["oof"],1-x["oof"],x["val"],x["seed_std"]]
        for pair in WATCH:
            vals.extend([x["pairs"][pair],x["pairs"][pair]/480])
        vals.extend(x["classerror"][q] for q in ("0","8","B","D"))
        vals.extend([x["flip"]["OOF"],x["flip"]["OfficialVal"],
                     x["mean_n_iter"],x["mean_fit_seconds"],x["warnings"],
                     target_under_5percent(x),all(c.values()),
                     ";".join(k for k,v in c.items() if not v)])
        assert len(vals)==len(cols),(len(vals),len(cols))
        rows.append(dict(zip(cols,vals)))
    with (a.out/"S3_alpha_comparison.csv").open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(rows)
    md=["# EXP-CMB-008 · 06 S3 L2 alpha 최적화 실측","",
        "> 판정: "+status+" / 신규 alpha 3개 × 3Seed × 4Fold=36개, 기준선 alpha=1e-4 기존 12개 재사용.","",
        "## 실험의 인과적 흐름",
        "S2의 (256,128)/(256,256) 두 층 모델은 전체 OOF를 소폭 개선하거나 감소시켰지만, 8→B 등 핵심 혼동을 악화시켰다. 따라서 (256,) 단층을 유지하고 모델 용량을 바꾸지 않은 상태에서 L2 규제 alpha만 비교한다.",
        "1e-5는 기본 1e-4보다 약한 규제, 1e-3과 1e-2는 더 강한 규제다. 다른 HP와 데이터는 고정했다.","",
        "## 정확도·4방향 실제 오분류율 (각 클래스 480회)",
        "| alpha | Hand-made OOF | OOF 오분류율 | OfficialVal | D→0 | B→8 | 8→B | 0→D | 모두 <5% | 탈락 사유 |",
        "|---:|---:|---:|---:|---:|---:|---:|---:|---|---|"]
    for row in rows:
        def err(n):return str(row[n])+" ("+pct(row[n+"_rate"])+")"
        md.append("| "+str(row["alpha"])+" | "+pct(row["oof_accuracy"])+" | "+pct(row["oof_error_rate"])+
          " | "+pct(row["official_val_accuracy"])+" | "+
          " | ".join(err(k.replace("->","_to_")) for k in WATCH)+" | "+
          ("충족" if row["target_4_under_5pct"] else "미달")+" | "+(row["failed_gates"] or "없음")+" |")
    md+=["","## 과적합·특정 클래스 붕괴 확인",
        "| alpha | 전체 0오류 | 8오류 | B오류 | D오류 | 최저 Writer 정확도 | Seed 표준편차 | OOF 최대 클래스 Flip | Val 최대 Flip | 수렴경고 |",
        "|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for k in ALPHAS:
        z=measured[k]
        md.append("| "+k+" | "+" | ".join(pct(z["classerror"][n]) for n in ("0","8","B","D"))+
          " | "+pct(min(z["writers"].values()))+" | "+pct(z["seed_std"])+
          " | "+pct(z["flip"]["OOF"])+" | "+pct(z["flip"]["OfficialVal"])+
          " | "+str(z["warnings"])+" |")
    md+=["","### 보류 Writer별 OOF 정확도 / 수렴",
        "| alpha | W41 | W42 | W43 | W44 | 평균 학습 반복 | 평균 시간(초) |",
        "|---:|---:|---:|---:|---:|---:|---:|"]
    for k in ALPHAS:
        z=measured[k]
        md.append("| "+k+" | "+" | ".join(pct(z["writers"][str(w)]) for w in WRITERS)+
          " | "+f'{z["mean_n_iter"]:.2f}'+" | "+f'{z["mean_fit_seconds"]:.2f}'+" |")
    md+=["","### 같은 이미지 교정/새 오답 수 (기준 alpha=1e-4 대비)",
        "| alpha | 기존 오류 교정 | 기존 정답 신규 오분류 | 순개선 |",
        "|---:|---:|---:|---:|"]
    for k in ALPHAS:
        if k=="1e-4":continue
        corr,ind=paired_metrics[k]
        md.append("| "+k+" | "+str(corr)+" | "+str(ind)+" | "+str(corr-ind)+" |")
    md+=["","## 확정·보류와 다음 단계",
        "- **4방향 각 5% 미만 및 가드레일 통과:** "+(str(primary) if primary else "없음"),
        "- **S4 학습률 검증의 기준 alpha:** "+winner,
        "- **선정 이유:** "+report["selection_reason"],
        "- **선정 규칙:** 목표 네 방향 모두 <5%를 우선 확인하고, 미달이면 D→0을 포함한 네 방향과 클래스 0/8/B/D의 오분류 비증가, Writer/OfficialVal/Flip 가드레일, 전체 OOF 최고 정확도를 적용한다. 0.20%p 이내는 D→0을 우선한다.",
        "- **과적합 주의:** Loss curve/n_iter/경고를 남겨도 개발용 OOF 반복 탐색으로 인한 과적합까지 배제할 수는 없다. 독립 Test는 사용하지 않는다.",
        "- **S4 다음 질문:** alpha와 단층 (256,)를 고정한 뒤 Adam learning_rate_init 0.0005,0.001,0.002의 수렴·오류 재배분 효과를 비교한다.",
        "- Test·교수님 Test·신규 480장 미사용.",""]
    (a.out/"S3_Stage_Results.md").write_text("\n".join(md),encoding="utf-8")
    print("S3_DECISION",json.dumps({"status":status,"selected_alpha":winner,"primary":primary,
         "fallback":fallback,"rows":rows},ensure_ascii=False),flush=True)

if __name__=="__main__":main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")